# P8-3 · Bài 10b · Giám sát và chi phí

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q fastapi prometheus_client

## Lý thuyết

## Bài tập 5: Log và metric cho API {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, time, uuid, logging
from fastapi import FastAPI, Request, Response
from prometheus_client import Histogram, Counter, generate_latest, CONTENT_TYPE_LATEST

app = FastAPI()
log = logging.getLogger("api"); logging.basicConfig(level=logging.INFO, format="%(message)s")
LATENCY = Histogram("request_latency_seconds", "Độ trễ request", ["path"], buckets=[0.1, 0.25, 0.5, 1, 2, 3, 5, 10])
ERRORS = Counter("request_errors_total", "Số request lỗi", ["path", "status"])

@app.middleware("http")
async def observe(request: Request, call_next):
    rid, t0 = str(uuid.uuid4()), time.perf_counter()
    response = await call_next(request)
    dt = time.perf_counter() - t0
    LATENCY.labels(request.url.path).observe(dt)
    if response.status_code >= 500:
        ERRORS.labels(request.url.path, str(response.status_code)).inc()
    log.info(json.dumps({"id": rid, "path": request.url.path, "status": response.status_code, "latency_s": round(dt, 3)}))
    response.headers["X-Request-ID"] = rid
    return response

@app.get("/metrics")
def metrics():
    return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)

# thử: gọi vài request rồi đọc metric như Prometheus sẽ đọc
from fastapi.testclient import TestClient
c = TestClient(app)
for _ in range(3): c.get("/metrics")
print([l for l in c.get("/metrics").text.splitlines() if l.startswith("request_latency_seconds_count")])

['request_latency_seconds_count{path="/metrics"} 3.0']
